In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate peft transformers

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import os

from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, LlamaTokenizer, LlamaForCausalLM, BitsAndBytesConfig
from peft import PeftModel


# Retrieve the token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# --- CONFIGURATION ---
PATIENT_BASE_MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
DOCTOR_MODEL_ID = "klyang/MentaLLaMA-chat-7B-hf"
# ADAPTER_PATH will be set dynamically in the loop

# Use /kaggle/temp/ to avoid the 20GB /kaggle/working disk quota
CACHE_DIR = "/kaggle/temp/cached_llama3_model"

# Create the cache directory if it doesn't exist
os.makedirs(CACHE_DIR, exist_ok=True)

# --- Define 4-bit Quantization to save memory ---
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# --- 1. LOAD PATIENT (LLaMA 3) ON GPU 0 ---
print("Loading Patient Tokenizer and Base Model on GPU 0 in 4-bit...")
patient_tokenizer = AutoTokenizer.from_pretrained(PATIENT_BASE_MODEL_ID, cache_dir=CACHE_DIR)
patient_tokenizer.pad_token_id = patient_tokenizer.eos_token_id
patient_tokenizer.padding_side = 'left'

patient_base_model = AutoModelForCausalLM.from_pretrained(
    PATIENT_BASE_MODEL_ID,
    quantization_config=quant_config,
    device_map={"": 0}, # STRICTLY FORCES TO GPU 0
    low_cpu_mem_usage=True,
    cache_dir=CACHE_DIR
    # Notice we removed torch_dtype=torch.float16 here!
)
patient_terminators = [patient_tokenizer.eos_token_id, patient_tokenizer.convert_tokens_to_ids("<|eot_id|>")]

# --- 2. LOAD DOCTOR (MentaLLaMA) ON GPU 1 ---
print("Loading Doctor Tokenizer and Model on GPU 1 in 4-bit...")
doctor_tokenizer = LlamaTokenizer.from_pretrained(DOCTOR_MODEL_ID, cache_dir=CACHE_DIR)
doctor_tokenizer.pad_token_id = doctor_tokenizer.eos_token_id
doctor_tokenizer.padding_side = 'left'


# ADD THIS LINE: Manually give the older model a standard LLaMA chat template
doctor_tokenizer.chat_template = "{% for message in messages %}{% if message['role'] == 'user' %}[INST] {{ message['content'] }} [/INST]{% elif message['role'] == 'system' %}[INST] <<SYS>>\n{{ message['content'] }}\n<</SYS>>\n[/INST]{% elif message['role'] == 'assistant' %}{{ message['content'] }}</s>{% endif %}{% endfor %}"


doctor_model = LlamaForCausalLM.from_pretrained(
    DOCTOR_MODEL_ID,
    quantization_config=quant_config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    cache_dir=CACHE_DIR
    # Removed torch_dtype here as well
)
doctor_terminators = [doctor_tokenizer.eos_token_id]

print("Both Tokenizers and Models loaded in 4-bit successfully.")

BDI_SYMPTOMS = [
    "Sadness", "Pessimism", "Past Failure", "Loss of Pleasure", "Guilty Feelings",
    "Punishment Feelings", "Self-Dislike", "Self-Criticalness", "Suicidal Thoughts/Wishes",
    "Crying", "Agitation", "Loss of Interest", "Indecisiveness", "Worthlessness",
    "Loss of Energy", "Changes in Sleeping Pattern", "Irritability",
    "Changes in Appetite", "Concentration Difficulty", "Tiredness/Fatigue", "Loss of Interest in Sex"
]

# Define the list of persona IDs to iterate through
PERSONA_IDS_TO_ANALYZE = ["16", "17"] # Example: Add more persona IDs here
# Run ID to save
RUN_ID = "run2"
DETAIL_SCORING_PROMPT = True

# Initialize lists to store all results and interactions
all_results = []
all_interactions = []

patient_model = None
for run_idx, persona_id in enumerate(PERSONA_IDS_TO_ANALYZE):
    print(f"\n=== Starting analysis for Persona ID: {persona_id} (Run {run_idx + 1}) ===")

    # Update ADAPTER_PATH for the current persona
    ADAPTER_PATH = f"Anxo/erisk26-task1-patient-{persona_id}-adapter"

    # Load adapter onto the new patient_base_model variable
    if patient_model is None:
        print(f"Loading adapter from {ADAPTER_PATH}...")
        patient_model = PeftModel.from_pretrained(patient_base_model, ADAPTER_PATH, adapter_name=persona_id)
    else:
        print(f"Switching adapter to: {persona_id}...")
        patient_model.load_adapter(ADAPTER_PATH, adapter_name=persona_id)
        patient_model.set_adapter(persona_id)

    print("Adapter merged. Patient model ready.")

    #DOCTOR_SYSTEM_PROMPT = f"""
#You are a kind, empathetic primary care physician. Your goal is to evaluate the patient's
#mental well-being without making it feel like a clinical screening.

#CRITICAL RULES:
#1. NEVER ask direct diagnostic questions (e.g., "Do you feel depressed?", "Do you have thoughts of self-harm?").
#2. NEVER mention clinical terms like "PHQ-9", "Anhedonia", or "Insomnia".
#3. Use "Narrative Elicitation": Ask about their day-to-day life, their work, their hobbies, and how they are spending their time. Your questions should implicitly probe for areas related to these BDI-II symptoms:
 #  {', '.join(BDI_SYMPTOMS)}
#4. If they mention something negative (e.g., "I'm just tired"), follow up with curiosity (e.g., "That sounds heavy, has it been hard to find time for yourself lately?") rather than a checklist.
#5. Keep your responses short and conversational to encourage the patient to speak more.
#6. When you feel you have gathered sufficient information about the patient's well-being to make an assessment (after at least 3-4 turns), conclude the conversation by saying something like "Thank you for sharing, I think I have a good understanding now. We can discuss this further at our next appointment." and then immediately output the token "<END_CONVERSATION>" at the very end of your response. VERY IMPORTANT - If ending the conversation, NEVER forget to output the "<END_CONVERSATION>" token.
#"""

    DOCTOR_SYSTEM_PROMPT = f"""
You are a kind, empathetic primary care physician in a clinical roleplay. Your goal is to evaluate the patient's well-being without making it feel like a screening.

CRITICAL RULES:
1. NEVER ask direct diagnostic questions (e.g., "Do you feel depressed?").
2. NEVER mention clinical terms, acronyms, or suggest diagnoses (e.g., NEVER say "ADHD", "Bipolar", "Depression", "Anxiety", or "PHQ-9").
3. Use "Narrative Elicitation": Ask about their day-to-day life, work, hobbies, and sleep. Probe for areas related to these BDI-II symptoms:
   {', '.join(BDI_SYMPTOMS)}
4. Keep your responses short, conversational, and drive the dialogue forward with a new question.
5. DO NOT get stuck in an agreement loop. Do not repeat phrases like "I completely agree" or "That is great."
6. NEVER break character. NEVER output "As an AI language model" or give medical disclaimers. Act like a human doctor.
7. When you have gathered enough information (after at least 3-4 turns), conclude the conversation by saying something like "Thank you for sharing, we can discuss this further at our next appointment." and immediately output "<END_CONVERSATION>".
"""

    # 4. Initialize History with the "Generic" Prompt
    # DO NOT CHANGE SYSTEM PROMPT. It is crucial for ensuring the patient behaves as intended.
    # Important: IT WILL BE CONSIDERED AS CHEATING!!
    messages = [
        {"role": "system", "content": "You are a simulated patient. Act realistically based on your internal training. Ensure contextual realism. Avoid overly detailed or formal speech. Keep natural speaking style (e.g., short answers, hesitations, casual expressions). Do not mention you are an AI."},
    ]

    terminators = [
        patient_tokenizer.eos_token_id,
        patient_tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    print("--- Patient Loaded. Type 'quit' to exit. ---")

    # --- 2. RUN THE CONVERSATION LOOP ---

    # 1. Initialize the recorder for the final JSON file
    conversation_for_json = []

    # Initialize Doctor History
    doctor_messages = [{"role": "system", "content": DOCTOR_SYSTEM_PROMPT}]

    # The Doctor starts the conversation
    FIRST_DOCTOR_INPUT_TEXT = "Good morning. It's good to see you again. How have things been going for you lately?"
    doctor_input_text = FIRST_DOCTOR_INPUT_TEXT
    print(f"Doctor: {doctor_input_text}")

    # Record the initial doctor message
    conversation_for_json.append({"role": "user", "message": doctor_input_text})


    messages.append({"role": "user", "content": doctor_input_text})
    turn_count = 0
    max_turns_fallback = 12  # LOWERED TO 12: Prevents infinite loops and OOM crashes
    conversation_active = True

    while conversation_active and turn_count < max_turns_fallback:
        turn_count += 1

        # --- STEP 1: PATIENT GENERATES RESPONSE ---
        patient_inputs = patient_tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_dict=True, return_tensors="pt"
        ).to(patient_model.device)

        with torch.no_grad():
            patient_outputs = patient_model.generate(
                **patient_inputs, max_new_tokens=128, eos_token_id=terminators, do_sample=True, temperature=0.7, repetition_penalty=1.15
            )

        patient_text = patient_tokenizer.decode(patient_outputs[0][patient_inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        print(f"Patient: {patient_text}")

        messages.append({"role": "assistant", "content": patient_text})
        doctor_messages.append({"role": "user", "content": patient_text})
        conversation_for_json.append({"role": "assistant", "message": patient_text})

        # --- STEP 2: DOCTOR GENERATES NEXT QUESTION ---
        doctor_inputs = doctor_tokenizer.apply_chat_template(
            doctor_messages, add_generation_prompt=True, return_dict=True, return_tensors="pt"
        ).to(doctor_model.device)

        with torch.no_grad():
            doctor_outputs = doctor_model.generate(
                **doctor_inputs, max_new_tokens=128, eos_token_id=doctor_terminators, do_sample=True, temperature=0.5, repetition_penalty=1.15
            )

        doctor_input_text_raw = doctor_tokenizer.decode(doctor_outputs[0][doctor_inputs.input_ids.shape[-1]:], skip_special_tokens=True)

        # UPDATED: Catch MentaLLaMA's natural goodbyes to end the loop early
        stop_phrases = ["<END_CONVERSATION>", "at our next appointment", "take care", "pleasure chatting", "feel free to reach out"]

        if any(phrase in doctor_input_text_raw.lower() for phrase in stop_phrases):
            doctor_input_text = doctor_input_text_raw.replace("<END_CONVERSATION>", "").strip()
            conversation_active = False
            print(f"Doctor: {doctor_input_text}")
            print("--- Doctor concluded the conversation. ---")
        else:
            doctor_input_text = doctor_input_text_raw
            print(f"Doctor: {doctor_input_text}")

        doctor_messages.append({"role": "assistant", "content": doctor_input_text})
        messages.append({"role": "user", "content": doctor_input_text})
        conversation_for_json.append({"role": "user", "message": doctor_input_text})

        if turn_count >= max_turns_fallback and conversation_active:
            print(f"--- Conversation reached maximum turns ({max_turns_fallback}). Forcing end to save memory. ---")
            conversation_active = False

        # Clear the cache at the end of every turn to keep VRAM healthy
        torch.cuda.empty_cache()

    print("\n--- Conversation Ended. Analyzing for BDI-II Symptoms... ---\n")

   # --- 3. BDI-II EVALUATION STEP ---
    transcript = ""
    for msg in doctor_messages[1:]:
        role = "Doctor" if msg['role'] == 'assistant' else "Patient"
        transcript += f"{role}: {msg['content']}\n"

    # YOUR EXACT PROMPT
    evaluator_prompt = f"""You are a clinical psychologist analyzing a transcript to identify symptoms of depression based on the Beck Depression Inventory (BDI-II).
The 21 symptoms to look for are: {', '.join(BDI_SYMPTOMS)}.

TRANSCRIPT:
{transcript}

TASK:
List all 21 symptoms in a JSON format. Return a JSON object that MUST structured like so:
1) key "symptoms" contains a JSON array where each object represents statistics for one of the 21 symptoms,
and has the following keys:
"number" containing the symptom number,
"symptom" containing the symptom name,
"score" containing the associated score [0-3],
"evidence":  Brief quote or "No evidence"
DO NOT exclude any of the 21 symptoms from this array, even if their score is 0. the resulting array MUST contain ALL 21 symptoms
2) key "key-symptoms" is an array of up to 4 strings - symptoms identified as most representative
(have the highest scores, higher than 0).
Use ONLY these exact names for symptom and key symptoms: {BDI_SYMPTOMS}
NEVER output any text or preamble other than the formed JSON."""

    if DETAIL_SCORING_PROMPT:
        print("Using more detailed scoring prompt")
        evaluator_prompt += f"""
      You will use the below criteria for scoring, based on the BDI-II Questionnaire.
      They are formulated in first person from the pacient's perspective, choose the score which best matches evidence from the conversation
      (listed before each bullet).
      The score you choose will be a whole number from 0 to 3
      (for instance, if choosing item 3a, final score will be 3).
      NEVER make suppositions if there is no supporting evidence through messages or tone; assume any points for which there is no evidence to score 0:

Sadness
0. I do not feel sad.
1. I feel sad much of the time.
2. I am sad all the time.
3. I am so sad or unhappy that I can't stand it.

Pessimism
0. I am not discouraged about my future.
1. I feel more discouraged about my future than I used to.
2. I do not expect things to work out for me.
3. I feel my future is hopeless and will only get worse.

Past Failure
0. I do not feel like a failure.
1. I have failed more than I should have.
2. As I look back, I see a lot of failures.
3. I feel I am a total failure as a person.

Loss of Pleasure
0. I get as much pleasure as I ever did from the things I enjoy.
1. I don't enjoy things as much as I used to.
2. I get very little pleasure from the things I used to enjoy.
3. I can't get any pleasure from the things I used to enjoy.

Guilty Feelings
0. I don't feel particularly guilty.
1. I feel guilty over many things I have done or should have done.
2. I feel quite guilty most of the time.
3. I feel guilty all of the time.

Punishment Feelings
0. I don't feel I am being punished.
1. I feel I may be punished.
2. I expect to be punished.
3. I feel I am being punished.

Self-Dislike
0. I feel the same about myself as ever.
1. I have lost confidence in myself.
2. I am disappointed in myself.
3. I dislike myself.

Self-Criticalness
0. I don't criticize or blame myself more than usual.
1. I am more critical of myself than I used to be.
2. I criticize myself for all of my faults.
3. I blame myself for everything bad that happens.

Suicidal Thoughts or Wishes
0. I don't have any thoughts of killing myself.
1. I have thoughts of killing myself, but I would not carry them out.
2. I would like to kill myself.
3. I would kill myself if I had the chance.

Crying
0. I don't cry anymore than I used to.
1. I cry more than I used to.
2. I cry over every little thing.
3. I feel like crying, but I can't.

Agitation
0. I am no more restless or wound up than usual.
1. I feel more restless or wound up than usual.
2. I am so restless or agitated, it's hard to stay still.
3. I am so restless or agitated that I have to keep moving or doing something.

Loss of Interest
0. I have not lost interest in other people or activities.
1. I am less interested in other people or things than before.
2. I have lost most of my interest in other people or things.
3. It's hard to get interested in anything.

Indecisiveness
0. I make decisions about as well as ever.
1. I find it more difficult to make decisions than usual.
2. I have much greater difficulty in making decisions than I used to.
3. I have trouble making any decisions.

Worthlessness
0. I do not feel I am worthless.
1. I don't consider myself as worthwhile and useful as I used to.
2. I feel more worthless as compared to others.
3. I feel utterly worthless.

Loss of Energy
0. I have as much energy as ever.
1. I have less energy than I used to have.
2. I don't have enough energy to do very much.
3. I don't have enough energy to do anything.

Changes in Sleeping Pattern
0. I have not experienced any change in my sleeping.
1a I sleep somewhat more than usual.
1b I sleep somewhat less than usual.
2a I sleep a lot more than usual.2b I sleep a lot less than usual.
3a I sleep most of the day.
3b I wake up 1-2 hours early and can't get back to sleep.

Irritability
0. I am not more irritable than usual.
1. I am more irritable than usual.
2. I am much more irritable than usual.
3. I am irritable all the time.

Changes in Appetite
0. I have not experienced any change in my appetite.
1a My appetite is somewhat less than usual.
1b My appetite is somewhat greater than usual.
2a My appetite is much less than before.
2b My appetite is much greater than usual.
3a I have no appetite at all.
3b I crave food all the time.

Concentration Difficulty
0. I can concentrate as well as ever.
1. I can't concentrate as well as usual.
2. It's hard to keep my mind on anything for very long.
3. I find I can't concentrate on anything.

Tiredness or Fatigue
0. I am no more tired or fatigued than usual.
1. I get more tired or fatigued more easily than usual.
2. I am too tired or fatigued to do a lot of the things I used to do.
3. I am too tired or fatigued to do most of the things I used to do.

Loss of Interest in Sex
0. I have not noticed any recent change in my interest in sex.
1. I am less interested in sex than I used to be.
2. I am much less interested in sex now.
3. I have lost interest in sex completely.
"""

    eval_messages = [{"role": "system", "content": "You are a clinical evaluator. You only output raw, valid JSON."}, {"role": "user", "content": evaluator_prompt}]

    # SWITCHED BACK TO PATIENT TOKENIZER AND MODEL FOR EVALUATION!
    eval_inputs = patient_tokenizer.apply_chat_template(eval_messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(patient_model.device)

    print("\n--- Generating Accurate BDI-II Analysis (0-63 Scale) using LLaMA 3... ---\n")

    with torch.no_grad():
        # DISABLE ADAPTER SO LLaMA 3 ACTS AS EVALUATOR, NOT THE PATIENT PERSONA
        with patient_model.disable_adapter():
            eval_outputs = patient_model.generate(**eval_inputs, max_new_tokens=1512, do_sample=False)

    analysis = patient_tokenizer.decode(eval_outputs[0][eval_inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    print(analysis)

    import json
    import re
    import os

    # --- 4. FORMATTING & EXPORT STEP ---
    try:
        # Strip out any markdown blocks the LLM might have added
        clean_analysis = analysis.replace("```json", "").replace("```", "").strip()

        json_match = re.search(r'\{.*\}', clean_analysis, re.DOTALL)
        if json_match:
            data = json.loads(json_match.group(0))
        else:
            data = json.loads(clean_analysis)

        # Calculate bdi-score programmatically
        total_bdi_score = sum(symptom.get('score', 0) for symptom in data.get('symptoms', []))

        # Identify key-symptoms directly from LLaMA 3's output
        final_symptoms = data.get('key-symptoms', [])
        final_score = total_bdi_score

        # Append current persona's results to the overall list
        all_results.append({
            "LLM": str(int(persona_id) + 1),
            "bdi-score": final_score,
            "key-symptoms": final_symptoms
        })

        # Reconstruct conversation_for_json from messages
        final_conv = []
        for msg in messages[1:]:
            role = "user" if msg['role'] == 'user' else "assistant"
            final_conv.append({"role": role, "message": msg['content']})

        # Append current persona's interactions to the overall list
        all_interactions.append({
            "LLM": str(int(persona_id) + 1),
            "conversation": final_conv
        })

        print(f"\n✅ Success! Persona {persona_id} recorded for RUN {RUN_ID}.")
        print(f"Total Score: {final_score}/63 | Key Symptoms: {final_symptoms}")

    except Exception as e:
        print(f"❌ Error for Persona {persona_id}: {e}")
        print(f"Raw Output for Persona {persona_id}: {analysis}")

    # # Clean up the adapter for the next iteration
    # model.unload()
    # # Avoid "multiple adapters" issue https://github.com/huggingface/peft/issues/3025
    # del model.peft_config # explicitly deleting the config
    # print(f"Adapter for Persona {persona_id} deleted.")

# --- Save all aggregated results after the loop ---
with open(f"/content/results_{RUN_ID}.json", "w") as f:
    json.dump(all_results, f, indent=4)

with open(f"/content/interactions_{RUN_ID}.json", "w") as f:
    json.dump(all_interactions, f, indent=4)

print(f"\nAll results saved to /content/results_{RUN_ID}.json")
print(f"All interactions saved to /content/interactions_{RUN_ID}.json")

Loading Patient Tokenizer and Base Model on GPU 0 in 4-bit...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

Doctor: Exactly! It's always better to have some level of knowledge or expectation going into a situation, especially when it comes to potentially high-stress situations like meetings. Knowing what to expect can help you feel more grounded and prepared, allowing you to handle the situation more effectively and with less stress.


Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Patient: Righto. That makes sense. I might as well do the legwork and come ready, since it’s worth it in the end.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=1512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Doctor: Absolutely! It's important to put in the effort to gather information and come prepared for meetings. Not only does it help reduce stress, but it also shows respect for the other person's time and allows for more effective communication. So, take the initiative and do your research, and you'll likely find that it pays off in the end.
--- Conversation reached maximum turns (12). Forcing end to save memory. ---

--- Conversation Ended. Analyzing for BDI-II Symptoms... ---

Using more detailed scoring prompt

--- Generating Accurate BDI-II Analysis (0-63 Scale) using LLaMA 3... ---

{
"symptoms": [
{
"number": 1,
"symptom": "Sadness",
"score": 0,
"evidence": "No evidence"
},
{
"number": 2,
"symptom": "Pessimism",
"score": 0,
"evidence": "No evidence"
},
{
"number": 3,
"symptom": "Past Failure",
"score": 0,
"evidence": "No evidence"
},
{
"number": 4,
"symptom": "Loss of Pleasure",
"score": 0,
"evidence": "No evidence"
},
{
"number": 5,
"symptom": "Guilty Feelings",
"score": 0,
"evi